In [1]:
import re
import numpy as np
import pandas as pd

In [5]:
forms = pd.read_csv('final-paraczech-package/paraczech/paraczech_forms.csv')
exponents = pd.read_csv('input-resources/exponents.csv')

In [ ]:
exponents['new_cell'] = exponents.apply(
    lambda row: '.'.join([row['cell'].lower().split('.')[1], row['cell'].lower().split('.')[0]]),
    axis=1
)

rules_dict = exponents.groupby('new_cell').apply(
    lambda df: list(zip(df['regex'], df['exponent']))
).to_dict()

In [6]:
exponents

,pos,cell,regex,exponent
0,noun,ACC.PL,^.*ý$,ý
1,noun,ACC.PL,^.*é$,é
2,noun,ACC.PL,^.*á$,á
3,noun,ACC.PL,^.*í$,í
4,noun,ACC.PL,^.*y$,y
...,...,...,...,...
774,verb,PASS.PL.F,^.*ny$,ny
775,verb,PASS.PL.F,^.*ty$,ty
776,verb,PASS.PL.N,^.*ena$,ena
777,verb,PASS.PL.N,^.*na$,na


In [ ]:
def extract_and_verify_suffix(word, cell, rules):
    if cell not in rules:
        return np.nan
    for pattern, expected in rules[cell]:
        match = re.search(pattern, str(word))
        if match:
            if str(word).endswith(str(expected)):
                return expected               
    return np.nan

In [ ]:
# based on nominative and instrumental in singular
def undeclensible(group):
    nom = group.loc[group['cell'] == 'sg.nom', 'orth_form']
    ins = group.loc[group['cell'] == 'sg.ins', 'orth_form']

    if nom.empty or ins.empty:
        return True
    
    return not bool(set(nom) & set(ins))


forms = forms.groupby('lexeme').filter(undeclensible)

In [ ]:
forms['extracted_exponent'] = forms.apply(
    lambda row: extract_and_verify_suffix(row['orth_form'], row['cell'], rules_dict), 
    axis=1
)

In [ ]:
forms.to_csv('final-paraczech-package/paraczech/paraczech_forms_new.csv')